# Model screening

This notebook provides a compact model-screening analysis for the runtime-forecasting component of the dimming-control framework.

It is intended as supporting material for the final paper pipeline. The main manuscript reproduction workflow is provided in:

`notebooks/03_reproduce_review_core_results.ipynb`

The purpose here is to compare a small set of candidate regressors using grouped battery-level train/test splits and show why Random Forest is a practical choice for the final simulation pipeline.

# Setup and paths

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

DATA_PATH = REPO_ROOT / "data" / "processed" / "engineered_metadata.csv"
TABLE_OUTDIR = REPO_ROOT / "outputs" / "tables"
TABLE_OUTDIR.mkdir(parents=True, exist_ok=True)

print("Repository root:", REPO_ROOT)
print("Input data:", DATA_PATH)
print("Table outputs:", TABLE_OUTDIR)

# Load processed feature table

In [ ]:
df = pd.read_csv(DATA_PATH)
print("Loaded:", df.shape)
display(df.head())

# Feature and target definition

The feature set matches the forecasting inputs used by the main simulation notebook.

In [ ]:
GROUP_COL = "battery_id"
TARGET = "cycle_duration"

FEATURES = [
    "avg_voltage", "voltage_drop", "voltage_gradient", "voltage_std",
    "avg_current", "max_current", "current_std",
    "initial_temp", "final_temp", "temp_rise",
    "energy_approx", "start_voltage", "end_voltage",
]

required = ["uid", GROUP_COL, TARGET] + FEATURES
missing = [col for col in required if col not in df.columns]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

df0 = df.copy()

if "type" in df0.columns:
    df0 = df0[df0["type"].astype(str).str.lower().eq("discharge")].copy()

for col in [TARGET] + FEATURES:
    df0[col] = pd.to_numeric(df0[col], errors="coerce")

df0[GROUP_COL] = df0[GROUP_COL].astype(str)
df0 = df0.dropna(subset=[GROUP_COL, TARGET] + FEATURES).reset_index(drop=True)

print("Prepared rows:", df0.shape[0])
print("Batteries:", df0[GROUP_COL].nunique())
print("Target mean:", round(df0[TARGET].mean(), 3))
print("Target std:", round(df0[TARGET].std(), 3))

# Candidate models

The screening is intentionally compact. It avoids future-work or unrelated deep-learning experiments and focuses on practical tabular regressors suitable for the paper pipeline.

In [ ]:
def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def make_models(seed=42):
    return {
        "Ridge": make_pipeline(
            StandardScaler(),
            Ridge(alpha=1.0)
        ),
        "SVR_RBF": make_pipeline(
            StandardScaler(),
            SVR(C=10.0, epsilon=0.1, kernel="rbf")
        ),
        "HistGradientBoosting": HistGradientBoostingRegressor(
            max_iter=300,
            learning_rate=0.05,
            random_state=seed
        ),
        "ExtraTrees": ExtraTreesRegressor(
            n_estimators=400,
            min_samples_leaf=2,
            random_state=seed,
            n_jobs=-1
        ),
        "RandomForest": RandomForestRegressor(
            n_estimators=400,
            min_samples_leaf=2,
            max_features="sqrt",
            random_state=seed,
            n_jobs=-1
        ),
    }

# Grouped battery-level model screening

Each split holds out entire batteries, preventing the same battery ID from appearing in both training and test sets.

In [ ]:
SPLIT_SEEDS = [0, 1, 2, 3, 4]
MODEL_SEED = 42
TEST_BATTERY_FRAC = 0.25

X = df0[FEATURES].to_numpy(dtype=float)
y = df0[TARGET].to_numpy(dtype=float)
groups = df0[GROUP_COL].to_numpy()

rows = []

for split_seed in SPLIT_SEEDS:
    splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=TEST_BATTERY_FRAC,
        random_state=split_seed
    )

    train_idx, test_idx = next(splitter.split(X, y, groups=groups))

    train_groups = set(groups[train_idx])
    test_groups = set(groups[test_idx])
    assert train_groups.isdisjoint(test_groups), "Battery leakage detected."

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    for model_name, model in make_models(seed=MODEL_SEED).items():
        model.fit(X_train, y_train)
        pred = model.predict(X_test)

        rows.append({
            "split_seed": split_seed,
            "model": model_name,
            "n_train": len(train_idx),
            "n_test": len(test_idx),
            "train_batteries": len(train_groups),
            "test_batteries": len(test_groups),
            "MAE": float(mean_absolute_error(y_test, pred)),
            "RMSE": rmse(y_test, pred),
            "R2": float(r2_score(y_test, pred)),
        })

screening_results = pd.DataFrame(rows)
display(screening_results.head())

# Summary table

In [ ]:
summary = (
    screening_results
    .groupby("model", as_index=False)
    .agg(
        MAE_mean=("MAE", "mean"),
        MAE_std=("MAE", "std"),
        RMSE_mean=("RMSE", "mean"),
        RMSE_std=("RMSE", "std"),
        R2_mean=("R2", "mean"),
        R2_std=("R2", "std"),
    )
    .sort_values("MAE_mean")
    .reset_index(drop=True)
)

display(summary)

out_path = TABLE_OUTDIR / "model_screening_summary.csv"
summary.to_csv(out_path, index=False)
print("Saved:", out_path)

# Model-screening plot

In [ ]:
ax = summary.plot(
    x="model",
    y="MAE_mean",
    kind="bar",
    yerr="MAE_std",
    figsize=(8, 4),
    legend=False,
    title="Grouped battery-level model screening"
)

ax.set_xlabel("Model")
ax.set_ylabel("MAE")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

# Interpretation

This notebook is a compact screening analysis rather than the final experimental protocol. The final paper pipeline uses Random Forest runtime forecasts with grouped battery splits and train-only out-of-fold residuals for uncertainty buffering.

Random Forest is retained in the main pipeline because it provides stable tabular performance, supports grouped cross-validation cleanly, and avoids unnecessary complexity for the downstream controller simulation.